# Solution: Encoding Categorical Features -- Tips Dataset

**Exercises from notebook `01_encoding_categorical_features.ipynb`**

Jingxu Li - 320230942071


## Exercise 1: One-Hot Encoding with drop='first'

Load seaborn.load_dataset('tips'). Apply OHE to sex, smoker, and day. How many columns does time add with and without drop='first'?


In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('../dataset/tips.csv')
print(f"Rows: {len(df)}, Columns: {df.columns.tolist()}")

ohe_drop = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
encoded_drop = ohe_drop.fit_transform(df[['sex', 'smoker', 'day']])
print(f"OHE with drop='first': {encoded_drop.shape[1]} columns")
print(f"  Features: {ohe_drop.get_feature_names_out()}")

ohe_no_drop = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_no_drop = ohe_no_drop.fit_transform(df[['sex', 'smoker', 'day']])
print(f"OHE without drop: {encoded_no_drop.shape[1]} columns")
print(f"  Features: {ohe_no_drop.get_feature_names_out()}")

ohe_time_drop = OneHotEncoder(drop='first', sparse_output=False)
ohe_time_nodrop = OneHotEncoder(sparse_output=False)
print(f"\n'time' with drop='first': {ohe_time_drop.fit_transform(df[['time']]).shape[1]} column")
print(f"'time' without drop: {ohe_time_nodrop.fit_transform(df[['time']]).shape[1]} columns")


Rows: 244, Columns: ['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']
OHE with drop='first': 5 columns
  Features: ['sex_Male' 'smoker_Yes' 'day_Sat' 'day_Sun' 'day_Thur']
OHE without drop: 8 columns
  Features: ['sex_Female' 'sex_Male' 'smoker_No' 'smoker_Yes' 'day_Fri' 'day_Sat'
 'day_Sun' 'day_Thur']

'time' with drop='first': 1 column
'time' without drop: 2 columns


## Exercise 2: Ordinal vs OHE for size

Apply ordinal encoding to size (treating it as ordered 1-6). Compare R2 of a linear regression predicting tip with ordinal vs OHE encoding for size.


In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

y = df['tip'].values
total_bill = df[['total_bill']].values

# Ordinal encoding for size
ord_enc = OrdinalEncoder(categories=[sorted(df['size'].unique())])
X_ord = np.hstack([ord_enc.fit_transform(df[['size']]), total_bill])

# One-hot encoding for size
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_ohe = np.hstack([ohe.fit_transform(df[['size']]), total_bill])

ord_pipe = Pipeline([('model', LinearRegression())])
ohe_pipe = Pipeline([('model', LinearRegression())])

ord_r2 = cross_val_score(ord_pipe, X_ord, y, cv=5, scoring='r2')
ohe_r2 = cross_val_score(ohe_pipe, X_ohe, y, cv=5, scoring='r2')

print(f"Ordinal Encoding -- R2: {ord_r2.mean():.4f} (+/- {ord_r2.std() * 2:.4f})")
print(f"One-Hot Encoding -- R2: {ohe_r2.mean():.4f} (+/- {ohe_r2.std() * 2:.4f})")

Ordinal Encoding -- R2: 0.4657 (+/- 0.2450)
One-Hot Encoding -- R2: 0.4135 (+/- 0.2829)


## Exercise 3: Verify Cross-Fold Target Encoding

Manually verify that the cross-fold target encoder does not use any target values from the validation fold during encoding (add a print statement inside the loop to inspect the fold sizes).


In [11]:
from sklearn.model_selection import KFold
from sklearn.base import BaseEstimator, TransformerMixin

class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, n_folds=5, smoothing=10.0, random_state=42):
        self.n_folds = n_folds
        self.smoothing = smoothing
        self.random_state = random_state

    def fit(self, X, y):
        self.global_mean_ = np.mean(y)
        return self

    def transform(self, X, y=None):
        return X  

    def fit_transform(self, X, y):
        X = np.array(X).ravel()
        y = np.array(y)
        self.fit(X.reshape(-1, 1), y)
        encoded = np.empty_like(y, dtype=float)
        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)

        for train_idx, val_idx in kf.split(X):
            print(f"  Fold -- train size: {len(train_idx)}, val size: {len(val_idx)}")
            y_train = y[train_idx]
            cat_stats = {}
            for cat in np.unique(X[train_idx]):
                mask = X[train_idx] == cat
                cat_count = mask.sum()
                cat_mean = y_train[mask].mean()
                smoothed = (cat_count * cat_mean + self.smoothing * self.global_mean_) / (cat_count + self.smoothing)
                cat_stats[cat] = smoothed

            for i in val_idx:
                cat = X[i]
                encoded[i] = cat_stats.get(cat, self.global_mean_)

        return encoded.reshape(-1, 1)

X_cat = df['day'].values
y = df['tip'].values

te = TargetEncoder()
print("Inspecting fold sizes during cross-fold target encoding:")
encoded_vals = te.fit_transform(X_cat, y)
print(f"\nEncoded values: {encoded_vals[:5].ravel()}")
print("[OK] No target values from validation fold were used during encoding.")


Inspecting fold sizes during cross-fold target encoding:
  Fold -- train size: 195, val size: 49
  Fold -- train size: 195, val size: 49
  Fold -- train size: 195, val size: 49
  Fold -- train size: 195, val size: 49
  Fold -- train size: 196, val size: 48

Encoded values: [3.19606557 3.20046179 3.19606557 3.23646838 3.19606557]
[OK] No target values from validation fold were used during encoding.


## Exercise 4: handle_unknown='ignore'

What happens to handle_unknown='ignore' in OHE when the test set contains a category not seen during training? Demonstrate this with a toy example.


In [14]:
from sklearn.preprocessing import OneHotEncoder

X_train = np.array(['cat', 'dog', 'bird']).reshape(-1, 1)
X_test = np.array(['cat', 'dog', 'elephant']).reshape(-1, 1)

try:
    ohe_default = OneHotEncoder(sparse_output=False)
    ohe_default.fit(X_train)
    ohe_default.transform(X_test)
    print("Default OHE -- no error (unexpected)")
except ValueError as e:
    print(f"Default OHE -- ERROR: {e}")

ohe_ignore = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_ignore.fit(X_train)
result = ohe_ignore.transform(X_test)
print(f"\nOHE with handle_unknown='ignore':")
print(f"  Test categories: {X_test.ravel()}")
print(f"  Encoded result:\n{result}")
print(f"  Feature names: {ohe_ignore.get_feature_names_out()}")
print("\n'elephant' is encoded as all-zeros (unknown -> all features = 0).")


Default OHE -- ERROR: Found unknown categories ['elephant'] in column 0 during transform

OHE with handle_unknown='ignore':
  Test categories: ['cat' 'dog' 'elephant']
  Encoded result:
[[0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 0.]]
  Feature names: ['x0_bird' 'x0_cat' 'x0_dog']

'elephant' is encoded as all-zeros (unknown -> all features = 0).
